# Colab ja Drive ühendus

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/My Drive/Colab Notebooks/')

Mounted at /content/drive


# EstNLTK

In [ ]:
pip install estnltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 91.2 MB/s eta 0:00:00


In [ ]:
'''
@InProceedings{laur-EtAl:2020:LREC,
  author    = {Laur, Sven  and  Orasmaa, Siim  and  Särg, Dage  and  Tammo, Paul},
  title     = {EstNLTK 1.6: Remastered Estonian NLP Pipeline},
  booktitle = {Proceedings of The 12th Language Resources and Evaluation Conference},
  month     = {May},
  year      = {2020},
  address   = {Marseille, France},
  publisher = {European Language Resources Association},
  pages     = {7154--7162},
  url       = {https://www.aclweb.org/anthology/2020.lrec-1.884}
}
'''

# Eesti WordNet

In [ ]:
from estnltk.wordnet import Wordnet
wn = Wordnet()

import json
import random
from collections import deque

This requires downloading resource 'estwordnet_2026-02-13' (size: 26M). Proceed with downloading? [Y/n] y


Unpacked resource into subfolder 'wordnet/estwn-et-2.7.0_2026-02-13/' of the resources dir.


# Seoste genereerimine

In [ ]:
# Leia võimalikud seosed
#   Hüperonüüm: ülemmõiste
#   Hüponüüm: alammõiste
#   Meronüüm: osa tervikust
#   Holonüüm: tervik osadest
#   Sünohulk: sünonüümid

def leia_suhted(sünohulk, suhe, otsingusõna, max_pikkus):
  sõnad = {}
  suhted_map = {
      "hüponüüm": sünohulk.hyponyms,
      "hüperonüüm": sünohulk.hypernyms,
      "meronüüm": sünohulk.meronyms,
      "holonüüm": sünohulk.holonyms
  }

  for s in suhted_map[suhe]:
    if not s.definition:
      continue

    sõna = s.name.split(".")[0].lower()

    if (
        otsingusõna in sõna or
        sõna in otsingusõna or
        len(sõna) > max_pikkus
        ):
      continue

    sõnad[sõna] = s.definition

  return sõnad

def leia_sünonüümid(sünohulk, otsingusõna, max_pikkus):
  sünonüümid = {}

  for lemma in sünohulk.lemmas:
    sõna = lemma.lower().strip()

    if (
      otsingusõna in sõna or
      sõna in otsingusõna or
      len(sõna) > max_pikkus
    ):
      continue

    sünonüümid[sõna] = ""

  return sünonüümid

# Vali 10 sõna ühest semantilise suhte kategooriast, kui seoseid on üle 7
def vali_suhted(sõnad, max_arv=7):
  if len(sõnad) <= max_arv:
    return sõnad

  valik = random.sample(list(sõnad.keys()), max_arv)
  return {v: sõnad[v] for v in valik}

# Salvesta seosed listi, tagasta seoste list
def suhted(sõna, pos):
  sünohulgad = wn[sõna]
  vasted = []

  nimisõnu = 0
  tegusõnu = 0

  sõna_väike = sõna.lower()
  max_pikkus = 16

  for s in sünohulgad:
    sügavus = leia_sügavus(s)
    if sügavus < 4:
      continue

    if not s.definition:
      continue

    sünonüümid = leia_sünonüümid(s, sõna_väike, max_pikkus)
    hüperonüümid = leia_suhted(s, "hüperonüüm", sõna_väike, max_pikkus)
    hüponüümid = leia_suhted(s, "hüponüüm", sõna_väike, max_pikkus)
    meronüümid = leia_suhted(s, "meronüüm", sõna_väike, max_pikkus)
    holonüümid = leia_suhted(s, "holonüüm", sõna_väike, max_pikkus)

    ülemine = len(hüperonüümid) + len(holonüümid)
    alumine = len(hüponüümid) + len(meronüümid)
    kokku = len(sünonüümid) + ülemine + alumine

    # Kui puudub piisav arv seoseid, katkesta sünohulga lisamine
    if kokku < 5 or len(sünonüümid) < 1 or ülemine < 1 or alumine < 1:
      continue

    kirjed = {
      "sõna": sõna,
      "tähendus": s.definition,
      "sünonüümid": vali_suhted(sünonüümid),
      "hüperonüümid": vali_suhted(hüperonüümid),
      "hüponüümid": vali_suhted(hüponüümid),
      "meronüümid": vali_suhted(meronüümid),
      "holonüümid": vali_suhted(holonüümid)
    }

    vasted.append(kirjed)

    # Loenda pärast kirje lisamist
    if s.pos.startswith("n"):
      nimisõnu += 1
    elif s.pos.startswith("v"):
      tegusõnu += 1

  return vasted, nimisõnu, tegusõnu

def leia_sügavus(sünohulk):
  järjekord = deque([(sünohulk, 0)])
  külastatud = set()

  while järjekord:
    tipp, sügavus = järjekord.popleft()

    if tipp in külastatud:
      continue
    külastatud.add(tipp)

    hüperonüümid = tipp.hypernyms

    if not hüperonüümid:
      return sügavus

    for h in hüperonüümid:
      järjekord.append((h, sügavus + 1))

  return 0

In [ ]:
sõna = "hing"
sünohulgad = wn[sõna]

for s in sünohulgad:
  print(leia_sügavus(s), s.definition)
  print("Hüperonüüm:")
  print(s.hypernyms)
  print(len(s.hypernyms), s.hypernyms)
  print()
  print("Hüponüüm:")
  print(s.hyponyms)
  print(len(s.hyponyms), s.hyponyms)
  print("-------")
  # print(s.pos)
  # print(s.name.split(".")[1])
  # print(s)


3 olend, keda iseloomustab kõrgelt arenenud aju, abstraktse mõtlemise võime ja artikuleeritud kõne, homo sapiens
Hüperonüüm:
["Synset('elusolend.n.01')"]
1 ["Synset('elusolend.n.01')"]

Hüponüüm:
["Synset('koba.n.01')", "Synset('karske.n.01')", "Synset('kasutaja.n.01')", "Synset('kerjaja.n.01')", "Synset('kiilakas.n.01')", "Synset('kirjaoskamatu.n.01')", "Synset('kirtsnina.n.01')", "Synset('kogeja.n.01')", "Synset('kohmerdis.n.01')", "Synset('kokkuleplane.n.01')", "Synset('konkurent.n.01')", "Synset('kröösus.n.01')", "Synset('külaline.n.01')", "Synset('kurakäeline.n.01')", "Synset('kuulmismäluga inimene.n.01')", "Synset('küünik.n.01')", "Synset('lambapea.n.01')", "Synset('libask.n.01')", "Synset('limpjalg.n.01')", "Synset('lollpea.n.02')", "Synset('looja.n.01')", "Synset('loru.n.01')", "Synset('lugeja.n.02')", "Synset('migrant.n.01')", "Synset('mittesuitsetaja.n.01')", "Synset('moderaator.n.01')", "Synset('möhkam.n.01')", "Synset('moraalijünger.n.01')", "Synset('mõtetelugeja.n.01')", "

In [ ]:
def loe_sagedused(failinimi):
  sõnad = []

  with open(failinimi, "r", encoding="utf-8") as f:
    next(f)

    for rida in f:
      tulbad = rida.strip().split("\t")
      if len(tulbad) < 2:
        continue
      sõna = tulbad[0]
      pos = tulbad[1]

      if ("S" in pos or "V" in pos) and len(sõna) <= 16:
        sõnad.append((sõna, pos))

  return sõnad

def salvesta_suhted(sagedus, failinimi):
  andmed = {}
  puuduvad = []
  kokku = len(sagedus)

  nimisõnu = 0
  tegusõnu = 0

  for i, (sõna, pos) in enumerate(sagedus, 1):
    try:
      suhe, n, v = suhted(sõna, pos)
      nimisõnu += n
      tegusõnu += v
      if suhe:
        andmed[sõna] = suhe
      else:
        puuduvad.append(sõna)
    except Exception as e:
      puuduvad.append(sõna)

    if i % 100 == 0:
      print(f"Töödeldud {i}/{kokku}")

  with open(failinimi, "w", encoding="utf-8") as f:
    json.dump(andmed, f, ensure_ascii=False, indent=2)

  mõistatusi = sum(len(v) for v in andmed.values())

  print("Valmis!")
  print(f"Salvestatud {len(andmed)} lemmat faili {failinimi}")
  print(f"Nimisõnu: {nimisõnu}; Tegusõnu: {tegusõnu}")
  print(f"Võrdluseks: {nimisõnu + tegusõnu}")
  print(f"Mõistatusi: {mõistatusi}")
  print(f"Puuduvad {len(puuduvad)} sõnad:")
  print(puuduvad)

sõnad = loe_sagedused("sagedused.txt")
print(f"Sobivaid sagedussõnastiku kirjeid kokku: {len(sõnad)}")
salvesta_suhted(sõnad, "relations_2026_05_11_korr.json")

Sobivaid sagedussõnastiku kirjeid kokku: 6799
Töödeldud 100/6799
Töödeldud 200/6799
Töödeldud 300/6799
Töödeldud 400/6799
Töödeldud 500/6799
Töödeldud 600/6799
Töödeldud 700/6799
Töödeldud 800/6799
Töödeldud 900/6799
Töödeldud 1000/6799
Töödeldud 1100/6799
Töödeldud 1200/6799
Töödeldud 1300/6799
Töödeldud 1400/6799
Töödeldud 1500/6799
Töödeldud 1600/6799
Töödeldud 1700/6799
Töödeldud 1800/6799
Töödeldud 1900/6799
Töödeldud 2000/6799
Töödeldud 2100/6799
Töödeldud 2200/6799
Töödeldud 2300/6799
Töödeldud 2400/6799
Töödeldud 2500/6799
Töödeldud 2600/6799
Töödeldud 2700/6799
Töödeldud 2800/6799
Töödeldud 2900/6799
Töödeldud 3000/6799
Töödeldud 3100/6799
Töödeldud 3200/6799
Töödeldud 3300/6799
Töödeldud 3400/6799
Töödeldud 3500/6799
Töödeldud 3600/6799
Töödeldud 3700/6799
Töödeldud 3800/6799
Töödeldud 3900/6799
Töödeldud 4000/6799
Töödeldud 4100/6799
Töödeldud 4200/6799
Töödeldud 4300/6799
Töödeldud 4400/6799
Töödeldud 4500/6799
Töödeldud 4600/6799
Töödeldud 4700/6799
Töödeldud 4800/6799
Töö